# Create z-score files

This notebook creates mean/std files for the paper. Resdidual coefficients are defined in a separated notebook.

In [1]:
import os
import yaml
import numpy as np
import xarray as xr

## ERA5 mean std

In [2]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_ERA5.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [3]:
N_levels = 11

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/dscale_3h/'
ds_example = xr.open_zarr(base_dir+'ERA5_SW_3h_2021.zarr')
level = np.array(ds_example['level'])

In [4]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_surf = list(set(varnames) - set(['U', 'V', 'T', 'Q']))
varname_upper = ['U', 'V', 'T', 'Q']

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

### Mean file

In [5]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean_6h = xr.Dataset(coords={"level": level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_mean_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_mean_6h[varname] = data_array

In [6]:
# ds_mean_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/ERA5_3h_mean_1980_2019_new.nc')

In [11]:
ds_3h = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_3h_mean_1980_2019.nc')
ds_3h_SW = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/ERA5_3h_mean_1980_2019_new.nc')

for varname in ds_3h.keys():
    print(f'=================== {varname} ===================')
    print(ds_3h[varname].values)
    print(ds_3h_SW[varname].values)

=================== VAR_2T ===================
290.3522173722694
289.3190163838651
=================== MSL ===================
101636.02926621184
101767.62018201096
=================== VAR_10V ===================
0.873942350769839
0.2583390309802013
=================== VAR_10U ===================
-0.26072289415013494
0.6721322646104195
=================== U ===================
[-0.34319156  0.03790973  2.74288486  6.42386911  8.82248812 11.59287245
 14.94868826 19.52588245 24.8600006  14.33904524  0.73781285]
[ 0.69234738  1.72959531  4.75712795  8.90847759 11.66208084 14.70850616
 18.28751199 22.84885978 27.74640536 16.19777515  2.71182151]
=================== V ===================
[ 0.89797592  1.94231892  2.17507869  0.39714699 -0.05320645 -0.00847903
  0.31194102  0.89837034  1.34006671  0.52205376 -0.09927631]
[ 0.32390665  0.71542663  0.57026672  0.167779    0.16155172  0.26555923
  0.50286358  0.82345859  0.31934799 -0.07824155 -0.02732414]
=================== T ================

### Std file

In [8]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={"level": level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [9]:
# ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/ERA5_3h_std_1980_2019_new.nc')

In [12]:
ds_3h = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_3h_std_1980_2019.nc')
ds_3h_SW = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/ERA5_3h_std_1980_2019_new.nc')

for varname in ds_3h.keys():
    print(f'=================== {varname} ===================')
    print(ds_3h[varname].values)
    print(ds_3h_SW[varname].values)

=================== VAR_2T ===================
10.253738201019486
9.488309830806458
=================== MSL ===================
669.4589901534357
636.8713723527925
=================== VAR_10V ===================
3.377895893264416
3.643928687894116
=================== VAR_10U ===================
2.3918602602295924
3.4974112599997866
=================== U ===================
[ 2.63879745  4.44049216  5.8813855   7.1103909   8.7521183  10.6184807
 12.99767326 16.19460319 18.32331105 12.12329474  8.48019539]
[ 4.07806245  5.94114142  6.63582477  8.22879448  9.90135141 11.9049623
 14.40205351 17.57505894 19.33069207 12.77483063  9.2559728 ]
=================== V ===================
[ 3.68121891  6.24882356  7.73042307  7.41146002  8.34179071  9.70744222
 11.67821738 14.3481787  14.83903406  7.16725947  3.41170739]
[ 4.25539723  6.05361678  6.54033184  7.38292939  8.49003599  9.93704805
 12.02016329 14.81420461 15.19861686  7.09132281  3.53316551]
=================== T ===================
[9

## WRF mean std

In [13]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [14]:
N_levels = 12

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/C404_land/'
ds_example = xr.open_zarr(base_dir+'C404_SW_1980.zarr')
level = np.array(ds_example['bottom_top'])

In [15]:
# ds_example

In [16]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_upper = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_P', 'WRF_Q_tot', 'WRF_Q_tot_05', 'WRF_W', 'WRF_Z']
varname_surf = list(set(varnames) - set(varname_upper))

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [17]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean_6h = xr.Dataset(coords={'bottom_top': level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_mean_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_mean_6h[varname] = data_array

In [18]:
# ds_mean_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/C404_mean_1980_2019_12lev.nc')

In [23]:
ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_mean_1980_2019_12lev.nc')
ds_SW = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/C404_mean_1980_2019_12lev.nc')

for varname in ds_SW.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_SW[varname].values)
        print(ds_new [varname].values)
    except:
        pass

=================== WRF_U10 ===================
0.5060486835874023
-0.3342867197628134
=================== WRF_SWDOWN ===================
201.72275474548658
213.64847886680724
=================== WRF_precip_025 ===================
0.14150668505464783
0.10322347222812078
=================== WRF_radar_composite ===================
3.0721068098719537
2.366863667811179
=================== WRF_MSLP ===================
100072.50711124798
98178.72452439737
=================== WRF_SP ===================
99847.13990700492
97764.58035076989
=================== WRF_TSLB ===================
290.923372549447
292.5199666445885
=================== WRF_precip ===================
0.16395056720326115
0.11036608059935528
=================== WRF_OLR ===================
251.54706935064638
259.2982143968527
=================== WRF_PWAT ===================
0.02396033814152682
0.023409101635668436
=================== WRF_TCC ===================
0.5694339991364911
0.47782415343560886
=================== WRF_V1

In [20]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [21]:
# ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/C404_std_1980_2019_12lev.nc')

In [24]:
ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_std_1980_2019_12lev.nc')
ds_SW = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/C404_std_1980_2019_12lev.nc')

for varname in ds_SW.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_SW[varname].values)
        print(ds_new [varname].values)
    except:
        pass

=================== WRF_U10 ===================
3.5344769299643986
2.378519653419203
=================== WRF_SWDOWN ===================
294.68660211272976
305.8034557250422
=================== WRF_precip_025 ===================
0.2998499457465394
0.2541216707417403
=================== WRF_radar_composite ===================
8.149960145234827
7.0917343159386075
=================== WRF_MSLP ===================
2190.1765299455747
3523.4324601980798
=================== WRF_SP ===================
2412.6465211915606
3880.5312815227153
=================== WRF_TSLB ===================
9.92303445447865
11.364951317463412
=================== WRF_precip ===================
1.257477507507448
1.0634725429860914
=================== WRF_OLR ===================
36.33821276670824
37.8922891543909
=================== WRF_PWAT ===================
0.014320123741500965
0.01355639332377419
=================== WRF_TCC ===================
0.49427273859660614
0.4985328685369063
=================== WRF_V10 ====